# Lesson 07 · Empirical Bayesian hierarchical modeling

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 8 (pp. 65–74)

**Question:** Two batters each have 30 hits in 100 at-bats; one bats left-handed and one right-handed. Who's more likely the better hitter, and does the answer depend on the era?

**Goal:** Extend beta-binomial regression with handedness, a spline over time and an interaction, and see how each changes the prior and the posterior.

**Contents**

1. Setup
2. Left- and right-handed batters
3. Batting averages over time
4. A spline for year
5. Interaction: has the lefty advantage changed?
6. Posteriors for the 30/100 batters
7. Uncertainty in the hyperparameters

**Tags:** `hierarchical-models` `beta-binomial-regression` `splines` `interactions` `handedness` `hyperparameter-uncertainty`

**Before you start:** Lesson 06. You'll need your `fit_bb_regression` function: paste it below the setup cell or import it from `my_eb.py`.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

career = eb_data.career(pitcher_rule=3, extra=True)   # adds `year` (mean season) and `bats`
print(f"{len(career):,} players")
print(career["bats"].value_counts(dropna=False))

# Splines: patsy's natural cubic spline cr() is closest to R's ns(). If patsy
# is missing, install it into the pymc env from an Anaconda Prompt:
#     conda install -n pymc -c conda-forge patsy
# or build a basis with scipy.interpolate.BSpline instead.
try:
    import patsy
    print("patsy", patsy.__version__)
except ImportError:
    print("patsy not installed (see note above)")

## 1. Setup

**Your turn.** Bring in your `fit_bb_regression` from Lesson 06. Refit the log(AB) model to confirm it works here.

In [ ]:
# Your code here

## 2. Left- and right-handed batters

`bats` is L, R or B (both: switch hitters). Players with missing handedness, mostly from the early years, are dropped for this part.

**Your turn.** How many players are L, R, B and missing? About 9 in 10 people in the general population are right-handed. How does baseball compare, and what might that suggest?

In [ ]:
# Your code here

**Your turn.** Drop missing `bats`. Build `X` with an intercept, `log(AB)`, `is_B` and `is_L`, so that right-handed is the baseline, and fit.

In [ ]:
# Your code here

> **Check:** lefty ≈ +0.010, switch ≈ −0.001 (not distinguishable from 0), log(AB) ≈ 0.0152.

**Your turn.** Plot the prior for L, R and B batters at AB = 1,000. Then, for a lefty and a righty who both hit 30%, plot the posteriors at AB = 10, 100, 1,000 and 10,000 (a 2×2 grid).

In [ ]:
# Your code here

In [ ]:
# Your code here

**Reflect.** Why do the left and right posteriors converge as AB grows?

*Your answer:*

## 3. Batting averages over time

**Your turn.** Make a box plot of raw averages by decade (use `year`, the mean season of a player's career; keep AB ≥ 500 to reduce noise).

In [ ]:
# Your code here

**Reflect.** Does a straight line in year look reasonable? What would you use instead?

*Your answer:*

## 4. A spline for year

The book uses a natural cubic spline with 5 degrees of freedom. With patsy, `patsy.dmatrix("cr(year, df=5) - 1", career)` builds the basis. The `- 1` removes the intercept, because the spline columns already span a constant. Cardprice notebook 07 builds B-spline bases by hand, and that works too.

**Your turn.** Build `X = [spline(year) | is_B | is_L | log(AB)]` (no separate intercept) and fit. Store the fitted coefficients.

In [ ]:
# Your code here

**Your turn.** For a hypothetical player with AB = 1,000, plot the prior median and the 2.5% and 97.5% quantiles against year (1880–2015), one line and band for L and one for R.

In [ ]:
# Your code here

## 5. Interaction: has the lefty advantage changed?

**Your turn.** Add interaction columns (each spline column × `is_L`, and × `is_B`), refit, and redraw the plot.

In [ ]:
# Your code here

**Reflect.** What happened to the left/right gap over time? Give one possible explanation from the game itself.

*Your answer:*

## 6. Posteriors for the 30/100 batters

**Your turn.** For a left- and a right-handed batter with 30 H / 100 AB in 1915, 1965 and 2015, compute the prior μ, α₀, β₀ and posterior α₁, β₁ from the interaction model. Plot the six posteriors.

In [ ]:
# Your code here

> **Check:** prior μ for L / R is about 0.230 / 0.211 in 1915, 0.208 / 0.201 in 1965, and 0.212 / 0.212 in 2015.

**Your turn.** Using a method from Lesson 05, compute P(lefty better than righty) in each of the three years.

In [ ]:
# Your code here

## 7. Uncertainty in the hyperparameters

Empirical Bayes treats every fitted coefficient as known exactly, even though each came from the same data (*double-dipping*). With a handful of parameters and 10,000 players that's harmless. With dozens of coefficients, some of them weakly determined, it starts to matter.

**Your turn.** Get standard errors for the interaction model's coefficients from the inverse Hessian. Which coefficients are least certain?

In [ ]:
# Your code here

**Reflect.** How many hyperparameters does the interaction model have? Which conclusions in this lesson would you trust less because the model ignores their uncertainty? Lesson 13 fits the same kind of model with full Bayes.

*Your answer:*

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 8.
- patsy splines: https://patsy.readthedocs.io/en/latest/spline-regression.html
- McElreath, *Statistical Rethinking* (2nd ed.), Chapter 4 (splines) and Chapter 13 (multilevel models).